# Generated Project Anatomy

Inspect the files produced when NRPy turns symbolic wave-equation expressions into a compiled project.

Navigation: [Index](../index.ipynb) | Previous: [NRPy Package Map](package_map.ipynb) | Next: [NRPy in Ten Minutes](../1-intro/ten_minute_overview.ipynb)

## Why This Matters
Symbolic equations are not yet simulations. Generated projects add build rules, parameter files, C functions, and diagnostics around the symbolic kernels.

## What You Will See
You will see a complete generated inventory, the parameter file, the Makefile, function prototypes, and representative generated C source files.

Prerequisite bridge: this notebook follows [NRPy Package Map](package_map.ipynb). Terms are defined here before they are used.

## Generate the Cartesian Wave Project
The real generator builds a project tree from NRPy registrations and generated C functions.

In [1]:
from pathlib import Path
import re
import subprocess
import sys
import tempfile

PROJECT_NAME = "wave_equation_cartesian"
workspace_manager = tempfile.TemporaryDirectory(prefix="nrpy_tutorial_cartesian_", dir=Path.cwd())
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / PROJECT_NAME

command = [sys.executable, "-m", "nrpy.examples.wave_equation_cartesian"]
print("generator command: python -m nrpy.examples.wave_equation_cartesian")
completed = subprocess.run(
    command,
    cwd=WORKSPACE,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
    timeout=300,
)
print(completed.stdout.replace(str(WORKSPACE), "<workspace>"))
assert PROJECT_DIR.is_dir()
print("project path: project/wave_equation_cartesian")

generator command: python -m nrpy.examples.wave_equation_cartesian


Finished! Now go into project/wave_equation_cartesian and type `make` to build, then ./wave_equation_cartesian to run.
    Parameter file can be found in wave_equation_cartesian.par

project path: project/wave_equation_cartesian


## Complete Inventory
The inventory is grouped by path names. Headers declare shared objects, C files implement behavior, `MoL` contains time stepping, and the parameter file supplies runtime values.

In [2]:
required = [
    "Makefile",
    "BHaH_function_prototypes.h",
    "rhs_eval.c",
    "wave_equation_cartesian.par",
]
for relative_path in required:
    assert (PROJECT_DIR / relative_path).is_file(), relative_path

print("complete project inventory:")
for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))

complete project inventory:
BHaH_defines.h
BHaH_function_prototypes.h
Makefile
MoL/MoL_free_intermediate_stage_gfs.c
MoL/MoL_malloc_intermediate_stage_gfs.c
MoL/MoL_step_forward_in_time.c
apply_bcs.c
cmdline_input_and_parfile_parser.c
commondata_struct_set_to_default.c
diagnostics.c
exact_solution_single_Cartesian_point.c
griddata_free.c
initial_data.c
intrinsics/simd_intrinsics.h
main.c
numerical_grids_and_timestep.c
params_struct_set_to_default.c
progress_indicator.c
rhs_eval.c
set_CodeParameters-nopointer.h
set_CodeParameters-simd.h
set_CodeParameters.h
wave_equation_cartesian.par


## Complete Representative Artifacts
These files are short enough to print in full and show the connection from parameters to build rules, prototypes, and the wave right-hand side.

In [3]:
for relative_path in [
    "wave_equation_cartesian.par",
    "Makefile",
    "BHaH_function_prototypes.h",
    "rhs_eval.c",
]:
    print(f"\n--- {relative_path} ---")
    print((PROJECT_DIR / relative_path).read_text(encoding="utf-8", errors="replace"))


--- wave_equation_cartesian.par ---
#### wave_equation_cartesian BH@H parameter file. NOTE: only commondata CodeParameters appear here ###
###########################
###########################
### Module: __main__
convergence_factor = 1.0        # (REAL)
diagnostics_output_every = 0.2  # (REAL)
###########################
###########################
### Module: nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData
sigma = 3.0                     # (REAL)
wavespeed = 1.0                 # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.MoLtimestepping.register_all
CFL_FACTOR = 0.5                # (REAL)
t_final = 8.0                   # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.diagnostics.progress_indicator
output_progress_every = 1       # (int)


--- Makefile ---
CC ?= gcc  # assigns the value CC to gcc only if environment variable CC is not already set

CFLAG

## Build and Run Evidence
A generated project is complete only if it builds and writes diagnostics. The run is shortened by editing the generated parameter file after inspection.

In [4]:
parfile = PROJECT_DIR / "wave_equation_cartesian.par"
par_text = parfile.read_text(encoding="utf-8")
par_text = par_text.replace("t_final = 8.0", "t_final = 0.2")
par_text = par_text.replace("diagnostics_output_every = 0.2", "diagnostics_output_every = 0.1")
par_text = par_text.replace("output_progress_every = 1", "output_progress_every = 1000000")
parfile.write_text(par_text, encoding="utf-8")
print(f"--- runtime {parfile.name} ---")
print(parfile.read_text(encoding="utf-8", errors="replace"))

build = subprocess.run(
    ["make", "-j2"],
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
    timeout=300,
)
print("build completed")
print("compiler output line count:", len(build.stdout.splitlines()))

for args in ([], ["2.0"]):
    run = subprocess.run(
        [f"./{PROJECT_NAME}", *args],
        cwd=PROJECT_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=True,
        timeout=90,
    )
    cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", run.stdout)
    cleaned = cleaned.replace(str(WORKSPACE), "<workspace>")
    label = "default resolution" if not args else "convergence factor 2.0"
    print(f"\nrun output ({label}):")
    for line in cleaned.splitlines():
        if line.strip():
            print(line.rstrip())

print("\ndiagnostic files:")
for diagnostic in sorted(PROJECT_DIR.glob("out0d-conv_factor*.txt")):
    print(diagnostic.name)
    text = diagnostic.read_text(encoding="utf-8", errors="replace")
    print(text)
    assert len(text.strip().splitlines()) >= 2

--- runtime wave_equation_cartesian.par ---
#### wave_equation_cartesian BH@H parameter file. NOTE: only commondata CodeParameters appear here ###
###########################
###########################
### Module: __main__
convergence_factor = 1.0        # (REAL)
diagnostics_output_every = 0.1  # (REAL)
###########################
###########################
### Module: nrpy.equations.wave_equation.WaveEquation_Solutions_InitialData
sigma = 3.0                     # (REAL)
wavespeed = 1.0                 # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.MoLtimestepping.register_all
CFL_FACTOR = 0.5                # (REAL)
t_final = 0.2                   # (REAL)
###########################
###########################
### Module: nrpy.infrastructures.BHaH.diagnostics.progress_indicator
output_progress_every = 1000000       # (int)



build completed
compiler output line count: 16



run output (default resolution):
It: 0 t=0.000 / 0.2 = 0.00% dt=1/6.4 | t/h=0.00 ETA 0h00m00s



run output (convergence factor 2.0):
It: 0 t=0.000 / 0.2 = 0.00% dt=1/12.8 | t/h=0.00 ETA 0h00m00s

diagnostic files:
out0d-conv_factor1.00.txt
0.000000e+00 0.000000e+00 0.000000e+00 3.991879e+00 3.991879e+00
1.562500e-01 4.061498e-08 2.140944e-05 3.983805e+00 3.983805e+00

out0d-conv_factor2.00.txt
0.000000e+00 0.000000e+00 0.000000e+00 3.997967e+00 3.997967e+00
7.812500e-02 6.435354e-10 1.354916e-06 3.995936e+00 3.995936e+00



## Interpreting the Output
The printed tree, source snippets, runtime parameter file, and diagnostic rows show the full loop: generator, source files, compiler, executable, and output. The shortened runtime input is the file used to produce the displayed diagnostics.

## Where This Leads
- [Installing NRPy and Running a First Standalone Project](install_and_run.ipynb)
- [C Function Registration](../1-intro/c_function.ipynb)
- [Start-to-Finish Cartesian Wave Project](../3-wave_equation/start_to_finish_wave_cartesian.ipynb)
- [BHaH Project Anatomy](../5-infrastructures/bhah_project_anatomy.ipynb)